# Pakistani Law RAG — Retrieval Pipeline
## BM25 + Semantic Search + RRF Fusion + CrossEncoder Re-ranking

This notebook builds and tests the full retrieval pipeline. By the end you will have
a single function `retrieve(query, strategy, top_k)` that:
1. Embeds the query with MiniLM
2. Runs BM25 keyword search over your chunk corpus
3. Runs semantic search against Pinecone
4. Fuses both ranked lists with Reciprocal Rank Fusion (RRF)
5. Re-ranks the fused top-20 with a CrossEncoder
6. Returns the top-K most relevant chunks

**Upload required:** Upload `chunks_fixed.json` and `chunks_recursive.json` to this Colab session
before running (Files panel on the left sidebar).

**Runtime:** T4 GPU recommended (for CrossEncoder speed).

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q pinecone sentence-transformers rank-bm25 tqdm

---
## Cell 2 — Configuration (EDIT THIS CELL)

In [ ]:
# ============================================================
#  FILL IN YOUR OWN VALUES
# ============================================================

PINECONE_API_KEY = "YOUR_PINECONE_API_KEY_HERE"
PINECONE_INDEX   = "pakistani-law"

NS_FIXED     = "fixed"
NS_RECURSIVE = "recursive"

EMBEDDING_MODEL    = "all-MiniLM-L6-v2"
CROSSENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# How many candidates each method returns before fusion
BM25_TOP_N      = 20
SEMANTIC_TOP_N  = 20

# How many results to re-rank with CrossEncoder
RERANK_TOP_N    = 20

# Final number of chunks returned to the LLM
FINAL_TOP_K     = 5

# RRF constant — 60 is standard, higher = more smoothing
RRF_K = 60

print("Config loaded.")

---
## Cell 3 — Imports

In [ ]:
import json
import torch
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---
## Cell 4 — Load Chunk Files

Make sure `chunks_fixed.json` and `chunks_recursive.json` are uploaded
to this session before running this cell.

In [ ]:
with open("chunks_fixed.json", "r", encoding="utf-8") as f:
    fixed_chunks = json.load(f)

with open("chunks_recursive.json", "r", encoding="utf-8") as f:
    recursive_chunks = json.load(f)

print(f"Fixed chunks loaded:     {len(fixed_chunks)}")
print(f"Recursive chunks loaded: {len(recursive_chunks)}")
print(f"\nSample fixed chunk keys: {list(fixed_chunks[0].keys())}")

---
## Cell 5 — Build BM25 Indexes

BM25 (Best Match 25) is a classical keyword-based ranking function.
It scores each chunk by how often query words appear in it,
weighted by how rare those words are across the whole corpus (TF-IDF style).

It runs entirely in memory — no GPU, no API calls.
We build one BM25 index per chunking strategy.

Tokenization: simple whitespace split. Good enough for English legal text.

In [ ]:
def build_bm25(chunks: list[dict]) -> BM25Okapi:
    """
    Build a BM25 index from a list of chunk dicts.
    Each chunk's text is tokenized by whitespace.
    """
    tokenized = [chunk['text'].lower().split() for chunk in chunks]
    return BM25Okapi(tokenized)


print("Building BM25 index for fixed chunks...")
bm25_fixed = build_bm25(fixed_chunks)
print("Done.")

print("Building BM25 index for recursive chunks...")
bm25_recursive = build_bm25(recursive_chunks)
print("Done.")

# Quick sanity check
test_scores = bm25_fixed.get_scores("murder punishment pakistan penal code".split())
top_idx     = np.argsort(test_scores)[::-1][:3]
print(f"\nBM25 sanity check — top 3 chunks for 'murder punishment':")
for idx in top_idx:
    print(f"  Score {test_scores[idx]:.3f} | {fixed_chunks[idx]['source']} | {fixed_chunks[idx]['text'][:100]}...")

---
## Cell 6 — Load Embedding Model and CrossEncoder

**Bi-encoder (MiniLM):** Embeds query to a vector → sent to Pinecone for semantic search.
Fast — runs once per query.

**CrossEncoder:** Takes (query, chunk) pairs and scores them jointly.
Much more accurate than cosine similarity but slower — only runs on the
top 20 candidates after fusion, not all 5000 chunks.

In [ ]:
print("Loading bi-encoder (MiniLM)...")
bi_encoder = SentenceTransformer(EMBEDDING_MODEL, device=device)
print("Bi-encoder loaded.")

print("\nLoading CrossEncoder...")
cross_encoder = CrossEncoder(CROSSENCODER_MODEL, device=device)
print("CrossEncoder loaded.")

print("\nConnecting to Pinecone...")
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)
stats = index.describe_index_stats()
print(f"Pinecone connected. Total vectors: {stats.total_vector_count}")

---
## Cell 7 — Individual Retrieval Functions

We define each retrieval step as a separate function so they can be
tested independently and mixed/matched for the ablation study.

In [ ]:
# ─────────────────────────────────────────────
#  A. BM25 Search
# ─────────────────────────────────────────────
def bm25_search(query: str,
                chunks: list[dict],
                bm25_index: BM25Okapi,
                top_n: int = 20) -> list[dict]:
    """
    Returns top_n chunks ranked by BM25 score.
    Each result dict includes the original chunk + bm25_score + bm25_rank.
    """
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_n]

    results = []
    for rank, idx in enumerate(top_indices):
        result = dict(chunks[idx])          # copy so we don't mutate original
        result['bm25_score'] = float(scores[idx])
        result['bm25_rank']  = rank + 1     # 1-indexed
        results.append(result)
    return results


# ─────────────────────────────────────────────
#  B. Semantic Search (via Pinecone)
# ─────────────────────────────────────────────
def semantic_search(query: str,
                    namespace: str,
                    top_n: int = 20) -> list[dict]:
    """
    Embeds the query with MiniLM, queries Pinecone, returns top_n chunks.
    Each result dict includes text + metadata + semantic_score + semantic_rank.
    """
    query_vec = bi_encoder.encode(query, convert_to_numpy=True).tolist()

    response = index.query(
        vector=query_vec,
        top_k=top_n,
        namespace=namespace,
        include_metadata=True
    )

    results = []
    for rank, match in enumerate(response.matches):
        result = {
            'id':             match.id,
            'text':           match.metadata.get('text', ''),
            'source':         match.metadata.get('source', ''),
            'year':           match.metadata.get('year', 0),
            'url':            match.metadata.get('url', ''),
            'strategy':       match.metadata.get('strategy', ''),
            'chunk_idx':      match.metadata.get('chunk_idx', 0),
            'semantic_score': float(match.score),
            'semantic_rank':  rank + 1
        }
        results.append(result)
    return results


print("Individual retrieval functions defined.")

---
## Cell 8 — Reciprocal Rank Fusion (RRF)

RRF merges two ranked lists into one without needing to normalize scores.

The formula for each document is:
```
rrf_score = 1/(rank_bm25 + K) + 1/(rank_semantic + K)
```
where K=60 is a smoothing constant that reduces the impact of very high ranks.

A document appearing at rank 1 in both lists gets the highest possible RRF score.
A document only found by one method still gets a partial score.
Documents not found by a method are simply absent from that list.

In [ ]:
def reciprocal_rank_fusion(bm25_results: list[dict],
                           semantic_results: list[dict],
                           k: int = 60,
                           top_n: int = 20) -> list[dict]:
    """
    Fuse BM25 and semantic results using RRF.

    Matching is done by chunk ID. Both result lists must have an 'id' field.
    Returns top_n chunks sorted by descending RRF score.
    """
    # Build a lookup: id -> chunk data
    # Semantic results are the source of truth for chunk text/metadata
    # BM25 results also have the full chunk but we prefer semantic metadata format
    all_chunks = {}

    for r in bm25_results:
        all_chunks[r['id']] = r
    for r in semantic_results:
        all_chunks[r['id']] = r   # overwrite — semantic has cleaner metadata

    # Build rank lookups
    bm25_ranks     = {r['id']: r['bm25_rank']     for r in bm25_results}
    semantic_ranks = {r['id']: r['semantic_rank'] for r in semantic_results}

    # Compute RRF score for every unique document seen by either method
    all_ids = set(bm25_ranks.keys()) | set(semantic_ranks.keys())
    rrf_scores = {}

    for doc_id in all_ids:
        score = 0.0
        if doc_id in bm25_ranks:
            score += 1.0 / (bm25_ranks[doc_id] + k)
        if doc_id in semantic_ranks:
            score += 1.0 / (semantic_ranks[doc_id] + k)
        rrf_scores[doc_id] = score

    # Sort by RRF score descending
    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    top_ids    = sorted_ids[:top_n]

    results = []
    for rank, doc_id in enumerate(top_ids):
        chunk = dict(all_chunks[doc_id])
        chunk['rrf_score'] = rrf_scores[doc_id]
        chunk['rrf_rank']  = rank + 1
        # Record which methods found this chunk (useful for analysis)
        chunk['found_by']  = []
        if doc_id in bm25_ranks:     chunk['found_by'].append('bm25')
        if doc_id in semantic_ranks: chunk['found_by'].append('semantic')
        results.append(chunk)

    return results


print("RRF function defined.")

---
## Cell 9 — CrossEncoder Re-ranking

The CrossEncoder takes each (query, chunk) pair and scores them *jointly*.
Unlike the bi-encoder (MiniLM) which embeds query and chunk separately,
the CrossEncoder attends to both simultaneously — much more accurate
but O(N) in the number of candidates, so we only run it on the top 20.

In [ ]:
def rerank_with_crossencoder(query: str,
                              candidates: list[dict],
                              top_k: int = 5) -> list[dict]:
    """
    Re-rank candidate chunks using a CrossEncoder.
    Returns top_k chunks sorted by CrossEncoder score descending.
    """
    if not candidates:
        return []

    # Build (query, chunk_text) pairs for the CrossEncoder
    pairs  = [(query, c['text']) for c in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)

    # Attach scores and sort
    for i, chunk in enumerate(candidates):
        chunk['crossencoder_score'] = float(scores[i])

    reranked = sorted(candidates,
                      key=lambda x: x['crossencoder_score'],
                      reverse=True)

    for rank, chunk in enumerate(reranked):
        chunk['final_rank'] = rank + 1

    return reranked[:top_k]


print("CrossEncoder re-ranking function defined.")

---
## Cell 10 — Master Retrieve Function

This is the single function your Gradio app will call.
It supports 3 modes for the ablation study:
- `semantic_only` — Pinecone semantic search, no BM25, no re-ranking
- `hybrid` — BM25 + Semantic + RRF, no CrossEncoder
- `hybrid_rerank` — BM25 + Semantic + RRF + CrossEncoder (best quality)

In [ ]:
import time

def retrieve(query: str,
             chunk_strategy: str = "fixed",
             retrieval_mode: str = "hybrid_rerank",
             top_k: int = 5) -> dict:
    """
    Master retrieval function.

    Args:
        query:            The user's question
        chunk_strategy:   'fixed' or 'recursive'
        retrieval_mode:   'semantic_only' | 'hybrid' | 'hybrid_rerank'
        top_k:            Number of final chunks to return

    Returns a dict with:
        chunks        — list of top_k chunk dicts
        context       — single string of all chunk texts (for LLM prompt)
        latency       — timing breakdown dict (ms)
        metadata      — retrieval stats for display
    """
    t_total_start = time.time()
    latency = {}

    # Select the right corpus and BM25 index
    if chunk_strategy == "fixed":
        chunks_corpus = fixed_chunks
        bm25_idx      = bm25_fixed
        namespace     = NS_FIXED
    else:
        chunks_corpus = recursive_chunks
        bm25_idx      = bm25_recursive
        namespace     = NS_RECURSIVE

    # ── Step 1: BM25 Search ─────────────────────────────────
    if retrieval_mode in ("hybrid", "hybrid_rerank"):
        t0 = time.time()
        bm25_results = bm25_search(query, chunks_corpus, bm25_idx, BM25_TOP_N)
        latency['bm25_ms'] = round((time.time() - t0) * 1000, 1)
    else:
        bm25_results = []
        latency['bm25_ms'] = 0

    # ── Step 2: Semantic Search ──────────────────────────────
    t0 = time.time()
    semantic_results = semantic_search(query, namespace, SEMANTIC_TOP_N)
    latency['semantic_ms'] = round((time.time() - t0) * 1000, 1)

    # ── Step 3: Fusion ───────────────────────────────────────
    if retrieval_mode in ("hybrid", "hybrid_rerank"):
        t0 = time.time()
        fused = reciprocal_rank_fusion(bm25_results, semantic_results,
                                       k=RRF_K, top_n=RERANK_TOP_N)
        latency['rrf_ms'] = round((time.time() - t0) * 1000, 1)
    else:
        # Semantic only — just use semantic results directly
        for rank, r in enumerate(semantic_results):
            r['rrf_rank'] = rank + 1
            r['found_by'] = ['semantic']
        fused = semantic_results[:top_k]
        latency['rrf_ms'] = 0

    # ── Step 4: CrossEncoder Re-ranking ─────────────────────
    if retrieval_mode == "hybrid_rerank":
        t0 = time.time()
        final_chunks = rerank_with_crossencoder(query, fused, top_k)
        latency['crossencoder_ms'] = round((time.time() - t0) * 1000, 1)
    else:
        final_chunks = fused[:top_k]
        for rank, c in enumerate(final_chunks):
            c['final_rank'] = rank + 1
        latency['crossencoder_ms'] = 0

    latency['total_ms'] = round((time.time() - t_total_start) * 1000, 1)

    # Build a single context string for the LLM prompt
    context_parts = []
    for i, chunk in enumerate(final_chunks):
        context_parts.append(
            f"[Source {i+1}: {chunk['source']} ({chunk['year']})]"
            f"\n{chunk['text']}"
        )
    context_string = "\n\n".join(context_parts)

    return {
        "chunks":   final_chunks,
        "context":  context_string,
        "latency":  latency,
        "metadata": {
            "query":            query,
            "chunk_strategy":   chunk_strategy,
            "retrieval_mode":   retrieval_mode,
            "bm25_candidates":  len(bm25_results),
            "semantic_candidates": len(semantic_results),
            "fused_candidates": len(fused),
            "final_returned":   len(final_chunks)
        }
    }


print("Master retrieve() function defined.")

---
## Cell 11 — Test All Three Retrieval Modes

Run the same query through all three modes and compare results.

In [ ]:
TEST_QUERY = "What is the punishment for murder under Pakistani law?"

modes = ["semantic_only", "hybrid", "hybrid_rerank"]

for mode in modes:
    print(f"\n{'='*60}")
    print(f"MODE: {mode.upper()}")
    print(f"{'='*60}")

    result = retrieve(TEST_QUERY, chunk_strategy="fixed", retrieval_mode=mode, top_k=3)

    print(f"Latency: {result['latency']}")
    print(f"Stats:   {result['metadata']}")
    print(f"\nTop 3 chunks returned:")
    for i, chunk in enumerate(result['chunks']):
        score_info = ""
        if 'crossencoder_score' in chunk:
            score_info = f"CrossEncoder={chunk['crossencoder_score']:.3f}"
        elif 'rrf_score' in chunk:
            score_info = f"RRF={chunk['rrf_score']:.4f}"
        elif 'semantic_score' in chunk:
            score_info = f"Semantic={chunk['semantic_score']:.4f}"
        found = chunk.get('found_by', [])
        print(f"  [{i+1}] {chunk['source']} | {score_info} | found_by={found}")
        print(f"       {chunk['text'][:150]}...")

---
## Cell 12 — Test With Multiple Legal Queries

Run a broader set of queries to confirm quality across different legal areas.

In [ ]:
TEST_QUERIES = [
    "What are the fundamental rights in Pakistan?",
    "What is the definition of terrorism under Pakistani law?",
    "How is a valid contract defined in Pakistan?",
    "What are the powers of the National Accountability Bureau?",
    "What is the procedure for filing an FIR?",
    "What are the rights of a wife under Muslim Family Laws?",
    "What is the punishment for theft under the Penal Code?",
    "How is evidence admitted in Pakistani courts?",
]

print(f"Testing {len(TEST_QUERIES)} queries in hybrid_rerank mode (fixed chunks)...\n")

for q in TEST_QUERIES:
    result = retrieve(q, chunk_strategy="fixed", retrieval_mode="hybrid_rerank", top_k=3)
    top = result['chunks'][0] if result['chunks'] else None
    if top:
        print(f"Q: {q}")
        print(f"   → Top source: {top['source']} | CE score: {top.get('crossencoder_score', 'N/A'):.3f} | "
              f"Latency: {result['latency']['total_ms']}ms")
    print()

---
## Cell 13 — Compare Fixed vs Recursive Chunking

Same query, same retrieval mode, different chunk strategies.
This data goes directly into your ablation study table.

In [ ]:
ABLATION_QUERY = "What constitutes an offence under the Anti-Terrorism Act?"

for strategy in ["fixed", "recursive"]:
    result = retrieve(ABLATION_QUERY,
                      chunk_strategy=strategy,
                      retrieval_mode="hybrid_rerank",
                      top_k=3)
    print(f"\n{'─'*55}")
    print(f"Strategy: {strategy.upper()} chunks")
    print(f"Latency:  {result['latency']}")
    for i, chunk in enumerate(result['chunks']):
        print(f"  [{i+1}] {chunk['source']} | CE={chunk.get('crossencoder_score', 0):.3f} | "
              f"found_by={chunk.get('found_by',[])}")
        print(f"       {chunk['text'][:200]}...")

---
## Cell 14 — Latency Benchmark

Run each mode 5 times and report average latency.
This goes into the 'latency and computational efficiency' section of your report.

In [ ]:
BENCH_QUERY = "What are the rights of an accused person in Pakistan?"
BENCH_RUNS  = 5

print(f"Benchmarking over {BENCH_RUNS} runs each...\n")

for mode in ["semantic_only", "hybrid", "hybrid_rerank"]:
    times = []
    breakdown = {'bm25_ms': [], 'semantic_ms': [], 'rrf_ms': [], 'crossencoder_ms': []}

    for _ in range(BENCH_RUNS):
        result = retrieve(BENCH_QUERY, "fixed", mode, top_k=5)
        times.append(result['latency']['total_ms'])
        for key in breakdown:
            breakdown[key].append(result['latency'][key])

    avg_total = round(sum(times) / len(times), 1)
    print(f"Mode: {mode}")
    print(f"  Avg total latency:    {avg_total} ms")
    for key, vals in breakdown.items():
        avg = round(sum(vals) / len(vals), 1)
        if avg > 0:
            print(f"  Avg {key:20s}: {avg} ms")
    print()

---
## Cell 15 — Save the Retrieve Function for Reuse

Save a standalone `retrieval.py` module. You will import this directly
in your Gradio app and evaluation notebooks — no duplication.

In [ ]:
retrieval_module = '''
"""
retrieval.py — Pakistani Law RAG Retrieval Pipeline
Import this module in your Gradio app and evaluation notebook.

Usage:
    from retrieval import load_retrieval_system, retrieve
    load_retrieval_system(pinecone_api_key, chunks_fixed_path, chunks_recursive_path)
    result = retrieve(query, chunk_strategy="fixed", retrieval_mode="hybrid_rerank", top_k=5)
"""

import json, time, numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone

# ── Globals (populated by load_retrieval_system) ────────────────
_bi_encoder      = None
_cross_encoder   = None
_pinecone_index  = None
_fixed_chunks    = None
_recursive_chunks = None
_bm25_fixed      = None
_bm25_recursive  = None

NS_FIXED     = "fixed"
NS_RECURSIVE = "recursive"
BM25_TOP_N   = 20
SEMANTIC_TOP_N = 20
RERANK_TOP_N = 20
RRF_K        = 60


def load_retrieval_system(pinecone_api_key: str,
                           index_name: str,
                           fixed_chunks_path: str,
                           recursive_chunks_path: str,
                           device: str = "cpu"):
    """Call once at app startup to initialise all components."""
    global _bi_encoder, _cross_encoder, _pinecone_index
    global _fixed_chunks, _recursive_chunks, _bm25_fixed, _bm25_recursive

    print("Loading bi-encoder...")
    _bi_encoder    = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    print("Loading CrossEncoder...")
    _cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

    print("Connecting to Pinecone...")
    pc = Pinecone(api_key=pinecone_api_key)
    _pinecone_index = pc.Index(index_name)

    print("Loading chunk files and building BM25 indexes...")
    with open(fixed_chunks_path,     "r", encoding="utf-8") as f:
        _fixed_chunks     = json.load(f)
    with open(recursive_chunks_path, "r", encoding="utf-8") as f:
        _recursive_chunks = json.load(f)

    _bm25_fixed     = BM25Okapi([c["text"].lower().split() for c in _fixed_chunks])
    _bm25_recursive = BM25Okapi([c["text"].lower().split() for c in _recursive_chunks])

    print("Retrieval system ready.")


def _bm25_search(query, chunks, bm25_index, top_n=20):
    scores     = bm25_index.get_scores(query.lower().split())
    top_indices = np.argsort(scores)[::-1][:top_n]
    results = []
    for rank, idx in enumerate(top_indices):
        r = dict(chunks[idx])
        r["bm25_score"] = float(scores[idx])
        r["bm25_rank"]  = rank + 1
        results.append(r)
    return results


def _semantic_search(query, namespace, top_n=20):
    vec = _bi_encoder.encode(query, convert_to_numpy=True).tolist()
    resp = _pinecone_index.query(vector=vec, top_k=top_n,
                                  namespace=namespace, include_metadata=True)
    results = []
    for rank, m in enumerate(resp.matches):
        results.append({
            "id": m.id,
            "text": m.metadata.get("text", ""),
            "source": m.metadata.get("source", ""),
            "year": m.metadata.get("year", 0),
            "url": m.metadata.get("url", ""),
            "strategy": m.metadata.get("strategy", ""),
            "chunk_idx": m.metadata.get("chunk_idx", 0),
            "semantic_score": float(m.score),
            "semantic_rank": rank + 1,
        })
    return results


def _rrf(bm25_results, semantic_results, k=60, top_n=20):
    all_chunks = {r["id"]: r for r in bm25_results}
    all_chunks.update({r["id"]: r for r in semantic_results})
    bm25_ranks     = {r["id"]: r["bm25_rank"]     for r in bm25_results}
    semantic_ranks = {r["id"]: r["semantic_rank"] for r in semantic_results}
    all_ids = set(bm25_ranks) | set(semantic_ranks)
    rrf_scores = {
        did: (1/(bm25_ranks.get(did, 1e9)+k)) + (1/(semantic_ranks.get(did, 1e9)+k))
        for did in all_ids
    }
    top_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:top_n]
    results = []
    for rank, did in enumerate(top_ids):
        c = dict(all_chunks[did])
        c["rrf_score"] = rrf_scores[did]
        c["rrf_rank"]  = rank + 1
        c["found_by"]  = (["bm25"] if did in bm25_ranks else []) + \
                          (["semantic"] if did in semantic_ranks else [])
        results.append(c)
    return results


def retrieve(query: str,
             chunk_strategy: str = "fixed",
             retrieval_mode: str = "hybrid_rerank",
             top_k: int = 5) -> dict:
    t0 = time.time()
    latency = {}

    chunks_corpus = _fixed_chunks    if chunk_strategy == "fixed" else _recursive_chunks
    bm25_idx      = _bm25_fixed      if chunk_strategy == "fixed" else _bm25_recursive
    namespace     = NS_FIXED         if chunk_strategy == "fixed" else NS_RECURSIVE

    # BM25
    t1 = time.time()
    bm25_r = _bm25_search(query, chunks_corpus, bm25_idx) if retrieval_mode != "semantic_only" else []
    latency["bm25_ms"] = round((time.time()-t1)*1000, 1)

    # Semantic
    t1 = time.time()
    sem_r = _semantic_search(query, namespace, SEMANTIC_TOP_N)
    latency["semantic_ms"] = round((time.time()-t1)*1000, 1)

    # Fusion
    t1 = time.time()
    fused = _rrf(bm25_r, sem_r) if retrieval_mode != "semantic_only" else sem_r[:RERANK_TOP_N]
    latency["rrf_ms"] = round((time.time()-t1)*1000, 1)

    # Rerank
    t1 = time.time()
    if retrieval_mode == "hybrid_rerank":
        pairs  = [(query, c["text"]) for c in fused]
        scores = _cross_encoder.predict(pairs, show_progress_bar=False)
        for i, c in enumerate(fused): c["crossencoder_score"] = float(scores[i])
        final = sorted(fused, key=lambda x: x["crossencoder_score"], reverse=True)[:top_k]
    else:
        final = fused[:top_k]
    latency["crossencoder_ms"] = round((time.time()-t1)*1000, 1)

    for rank, c in enumerate(final): c["final_rank"] = rank + 1
    latency["total_ms"] = round((time.time()-t0)*1000, 1)

    context = "\\n\\n".join(
        f"[Source {i+1}: {c[\'source\']} ({c[\'year\']})]\'\\n\'{c[\'text\']}"
        for i, c in enumerate(final)
    )

    return {
        "chunks":   final,
        "context":  context,
        "latency":  latency,
        "metadata": {
            "query": query, "chunk_strategy": chunk_strategy,
            "retrieval_mode": retrieval_mode, "final_returned": len(final)
        }
    }
'''

with open("retrieval.py", "w", encoding="utf-8") as f:
    f.write(retrieval_module)

print("retrieval.py saved. Download this file — you will need it for the Gradio app.")

from google.colab import files
files.download("retrieval.py")

---
## Cell 16 — Summary

Confirm everything is working before moving to Notebook 3 (LLM generation + evaluation).

In [ ]:
print("╔══════════════════════════════════════════════════╗")
print("║       RETRIEVAL PIPELINE — COMPLETE              ║")
print("╠══════════════════════════════════════════════════╣")
print("║  ✓ BM25 indexes built (fixed + recursive)        ║")
print("║  ✓ Semantic search via Pinecone                  ║")
print("║  ✓ RRF fusion implemented                        ║")
print("║  ✓ CrossEncoder re-ranking implemented           ║")
print("║  ✓ 3 retrieval modes: semantic/hybrid/full       ║")
print("║  ✓ Latency benchmarks recorded                   ║")
print("║  ✓ retrieval.py saved for reuse                  ║")
print("╠══════════════════════════════════════════════════╣")
print("║  Next: Notebook 3 — LLM Generation + Evaluation  ║")
print("╚══════════════════════════════════════════════════╝")